# PaginaVox

⚠️ Avant de lancer : dans Colab, va dans **Exécution > Modifier le type d'exécution > GPU**.

Sur "Easy AI", nous simplifions l'IA. Nos tutoriels vidéo sont conçus pour vous montrer pas à pas comment installer et utiliser différentes IA. Que vous soyez un passionné de technologie, un étudiant en informatique, ou simplement curieux de savoir comment l'IA peut vous être utile, nos vidéos sont là pour vous guider.

[Pour voir les tutos ](https://www.youtube.com/channel/UC-Of5gIC5_xPz6O-1YQ1aGw)

---
# J'ai créé cet outil pour un usage d'apprentissage uniquement, et non pour un usage commercial.
---

In [ ]:
#@title Vérification GPU
# Vérification GPU
import torch

print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("OK!")
else:
    print("Aucun GPU détecté. Active un GPU dans Colab pour Qwen TTS.")

In [ ]:
#@title 1. Installation des dépendances
!python -m pip install -q -U gradio qwen-tts soundfile numpy openai-whisper

In [ ]:
#@title 2. Création des fichiers du projet
from pathlib import Path
import base64

BASE_DIR = Path("/content/PaginaVox")
GRADIO_DIR = BASE_DIR / "gradio"
ASSETS_DIR = GRADIO_DIR / "assets"

for folder in [
    BASE_DIR,
    GRADIO_DIR,
    ASSETS_DIR,
    BASE_DIR / "audio",
    BASE_DIR / "txt",
    BASE_DIR / "output",
    BASE_DIR / "profiles",
]:
    folder.mkdir(parents=True, exist_ok=True)

APP_PY_B64 = "..."
MAIN_PY_B64 = "..."

(GRADIO_DIR / "app.py").write_text(
    base64.b64decode(APP_PY_B64).decode("utf-8"),
    encoding="utf-8"
)

(BASE_DIR / "main.py").write_text(
    base64.b64decode(MAIN_PY_B64).decode("utf-8"),
    encoding="utf-8"
)

print("Projet créé dans :", BASE_DIR)
print("Application Gradio :", GRADIO_DIR / "app.py")
print("Fichier principal :", BASE_DIR / "main.py")
print("Dossier de sortie autorisé par Gradio :", BASE_DIR / "output")

In [ ]:
#@title 3. Vérification rapide du patch `allowed_paths`
from pathlib import Path

app_text = Path("/content/PaginaVox/gradio/app.py").read_text(encoding="utf-8")
print(app_text[app_text.rfind('if __name__ == "__main__":'):])

Colab affichera un lien du type :

`Running on public URL: https://....gradio.live`

Clique sur ce lien pour ouvrir ton application.

In [ ]:
#@title 4. Lancement de PaginaVox
%cd /content/PaginaVox/gradio
!python app.py



---



## Notes utiles

- Les fichiers audio générés seront dans `/content/PaginaVox/output`.
- Les profils de voix clonée seront dans `/content/PaginaVox/profiles`.
- Les audios de référence seront dans `/content/PaginaVox/audio`.
- À chaque nouveau runtime Colab, il faut relancer les cellules, car `/content` est temporaire.